In [1]:
from pathlib import Path
from pptx import Presentation
from docx import Document
from unstructured.partition.auto import partition
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests
# Add these to existing imports
from datetime import datetime

c:\Users\zyzai\miniconda3\envs\diss_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def extract_text_from_pptx(file_path):
    try:
        prs = Presentation(file_path)
        text = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
        return "\n".join(text)
    except Exception:
        return None

def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception:
        return None

def extract_text_with_unstructured(file_path):
    try:
        elements = partition(file_path=file_path)
        return "\n".join([str(el) for el in elements])
    except Exception:
        return None

def extract_text(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == ".pptx":
        text = extract_text_from_pptx(file_path)
    elif ext == ".docx":
        text = extract_text_from_docx(file_path)
    elif ext == ".pdf":
        text = extract_text_with_unstructured(file_path)
    else:
        text = None

    return text or extract_text_with_unstructured(file_path) or ""


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

base_path = Path("Diss_doc")  # 🔁 replace with actual path

all_chunks = []

for folder in base_path.iterdir():
    if folder.is_dir():
        for file in folder.rglob("*"):
            if file.suffix.lower() in [".pptx", ".docx", ".pdf"]:
                raw_text = extract_text(file)
                if raw_text:
                    chunks = text_splitter.split_text(raw_text)
                    for chunk in chunks:
                        all_chunks.append({
                            "content": chunk,
                            "source": f"{folder.name}/{file.name}"
                        })


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in all_chunks]
metas = [c["source"] for c in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings))


Batches: 100%|██████████| 82/82 [00:05<00:00, 16.15it/s]


In [5]:
# ⬇️ ADD NEW: Setup second FAISS index for chat history
history_texts = []
history_embeddings = []
history_index = faiss.IndexFlatIP(384)  # same dim as MiniLM

def store_history_in_index(chat_history):
    global history_texts, history_embeddings, history_index
    new_texts = []
    for turn in chat_history:
        if turn["role"] == "user":
            new_texts.append(f"Client: {turn['content']}")
        else:
            new_texts.append(f"Consultant: {turn['content']}")

    if new_texts:
        new_embeddings = model.encode(new_texts, normalize_embeddings=True)
        history_texts.extend(new_texts)
        history_embeddings.extend(new_embeddings)
        history_index.add(np.array(new_embeddings))

def search_history(query, top_k=3):
    if not history_texts:
        return []

    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = history_index.search(np.array(query_vec), top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append((history_texts[idx], f"chat_turn_{idx}"))
    return results

In [6]:
def search(query, top_k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)
    
    for i, idx in enumerate(indices[0]):
        print(f"\n--- Match {i+1} ---")
        print(f"Cosine Similarity: {scores[0][i]:.4f}")
        print(f"Source: {metas[idx]}")
        print(f"Content:\n{texts[idx][:500]}...")


In [7]:
# ---- History memory (vector) ----
history_texts = []
history_index = faiss.IndexFlatIP(384)  # all-MiniLM-L6-v2 dimension

def store_history_in_index(turns):
    # turns: list[{"role": "...", "content": "..."}]
    if not turns:
        return
    new_texts = [
        ("Client: " + t["content"]) if t["role"] == "user" else ("Consultant: " + t["content"])
        for t in turns
    ]
    new_embs = model.encode(new_texts, normalize_embeddings=True)
    history_texts.extend(new_texts)
    history_index.add(np.array(new_embs))

def search_history(query, top_k=3):
    if not history_texts or history_index.ntotal == 0:
        return []
    q = model.encode([query], normalize_embeddings=True)
    scores, idxs = history_index.search(np.array(q), top_k)
    return [(history_texts[i], f"chat_turn_{i}") for i in idxs[0] if i != -1]


In [8]:
search("project discovery phase for a fintech client")



--- Match 1 ---
Cosine Similarity: 0.5954
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Key: Scope of discovery work
Proposed team & commercials 
[ ‹#› ]
We are proposing the following team structure for this initial phase, to mobilise the programme, demonstrate a repeatable approach that can be both scaled and utilised in stakeholder engagement. 
Delivery of this initial phase is estimated at £57k (excluding VAT). Our day rates are based on the current Informa-Clarasys rate card....

--- Match 2 ---
Cosine Similarity: 0.5523
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Our team will work with Procurement, Technology & Business Owners on the Discovery. Additionally, we will work in close collaboration with the wider programme team; focusing on the scope of WP2 over a seven week discovery period (start date to be confirmed). We will also provide supportin

In [9]:
def query_mistral(prompt, model="mistral:latest"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()['response']


In [10]:
# chat_history = []

# def get_expanded_query(user_input, chat_history, n_prev=1):
#     history = [
#         msg["content"]
#         for msg in chat_history[-2 * n_prev : -1]
#         if msg["role"] in ("user", "assistant")
#     ]
#     # return "\n".join(history + [user_input])
def get_expanded_query(user_input, history, n_prev=1):
    prev = [m["content"] for m in history[-2*n_prev:] if m["role"] in ("user","assistant")]
    return "\n".join(prev + [user_input])



In [11]:
# def build_prompt(history, context_chunks_with_sources, mode="initial"):
#     context_text = "\n<doc>\n".join(
#         f"[{src}]\n{chunk}" for chunk, src in context_chunks_with_sources
#     ) + "\n</doc>"

#     conversation = ""
#     for turn in history:
#         role = "Client" if turn["role"] == "user" else "Consultant"
#         conversation += f"{role}: {turn['content']}\n"

#     if mode == "initial":
#         prompt = f"""You are a senior consultant leading the discovery phase of a client project.

# You are given excerpts from past project documents. These are strictly your ONLY source of information. Do NOT use any external knowledge.

# Each excerpt starts with a filename in [brackets], followed by content. Stay within the facts only. Do NOT guess, assume, generalize, or invent anything that is not explicitly written.

# --- START OF CONTEXT ---

# {context_text}

# --- END OF CONTEXT ---

# Previous conversation:
# {conversation}

# Your task:

# - List all clear, stated client requirements from the request (not inferred ones).
# - Find similar past projects based only on visible matches in the excerpts.
# - For each similar project, include:
#   - The [filename]
#   - A short fact-based summary of what was done
# - Mention tools, industries, or outcomes only if they appear directly in the context.
# - Do NOT fabricate greetings, summaries, advice, or project plans.
# - Output strictly in this bullet list format:
#   - [filename]: short factual statement
#   - identify the technologies used in similar previous projects
#   - industries involved
#   - a basic timeframe
# """
#     elif mode == "followup":
#         prompt = f"""You are continuing a discussion with a senior consultant.

# You have access to project document excerpts below. You must use ONLY this context to answer follow-up questions. Do NOT use any external knowledge.

# --- CONTEXT ---

# {context_text}

# --- END ---

# Conversation so far:
# {conversation}

# Latest client question:
# {history[-1]['content']}

# Your response:
# - Stay within the bounds of the context.
# - Answer only what is asked.
# - Be factual and concise.
# - Do not repeat the original list of requirements or summaries unless explicitly requested.
# """

#     return prompt
def build_prompt(history, context_chunks_with_sources, latest_user_message, mode="initial"):
    context_text = "\n<doc>\n".join(
        f"[{src}]\n{chunk}" for chunk, src in context_chunks_with_sources
    ) + "\n</doc>"

    conversation = ""
    for turn in history:
        role = "Client" if turn["role"] == "user" else "Consultant"
        conversation += f"{role}: {turn['content']}\n"

    if mode == "initial":
        prompt = f"""You are a senior consultant leading the discovery phase of a client project.

# You are given excerpts from past project documents. These are strictly your ONLY source of information. Do NOT use any external knowledge.

# Each excerpt starts with a filename in [brackets], followed by content. Stay within the facts only. Do NOT guess, assume, generalize, or invent anything that is not explicitly written.

# --- START OF CONTEXT ---

# {context_text}

# --- END OF CONTEXT ---

# Previous conversation:
# {conversation}

# Your task:

# - List all clear, stated client requirements from the request (not inferred ones).
# - Find similar past projects based only on visible matches in the excerpts.
# - For each similar project, include:
#   - The [filename]
#   - A short fact-based summary of what was done
# - Mention tools, industries, or outcomes only if they appear directly in the context.
# - Do NOT fabricate greetings, summaries, advice, or project plans.
# - Output strictly in this bullet list format:
#   - [filename]: short factual statement
#   - identify the technologies used in similar previous projects
#   - industries involved
#   - a basic timeframe
"""
    elif mode == "followup":
        prompt = f"""You are continuing a discussion with a senior consultant.

# You have access to project document excerpts below. You must use ONLY this context to answer follow-up questions. Do NOT use any external knowledge.

# --- CONTEXT ---

# {context_text}

# --- END ---

# Conversation so far:
# {conversation}

# Latest client question:
# {history[-1]['content']}

# Your response:
# - Stay within the bounds of the context.
# - Answer only what is asked.
# - Be factual and concise.
# - Do not repeat the original list of requirements or summaries unless explicitly requested.
"""
    return prompt


In [12]:
# def rag(query, history, top_k=10, min_score=0.5):
#     expanded_query = get_expanded_query(query, chat_history)
#     query_vec = model.encode([expanded_query], normalize_embeddings=True)

#     scores, indices = index.search(np.array(query_vec), top_k)

#     context_chunks_with_sources = []
#     for i, idx in enumerate(indices[0]):
#         score = scores[0][i]
#         if score >= min_score:
#             chunk = texts[idx]
#             source = metas[idx]
#             context_chunks_with_sources.append((chunk, source))

#     # ⬇️ ADD: Fetch relevant chat history and merge into context
#     history_chunks = search_history(query)
#     context_chunks_with_sources.extend(history_chunks)

#     if not context_chunks_with_sources:
#         return "Not enough relevant context found."

#     mode = "initial" if len(chat_history) < 2 else "followup"
#     prompt = build_prompt(chat_history, context_chunks_with_sources, mode=mode)

#     response = query_mistral(prompt)
    
#     # Update history list
#     chat_history.append({"role": "user", "content": query})
#     chat_history.append({"role": "assistant", "content": response})

#     # ⬇️ Store history in embedding index
#     store_history_in_index([{"role": "user", "content": query}, {"role": "assistant", "content": response}])

#     return response

def rag(query, history, top_k=10, min_score=0.5):
    expanded_query = get_expanded_query(query, history)
    qv = model.encode([expanded_query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(qv), top_k)

    context_chunks_with_sources = []
    for i, idx in enumerate(idxs[0]):
        if scores[0][i] >= min_score:
            context_chunks_with_sources.append((texts[idx], metas[idx]))

    # add relevant chat memory
    context_chunks_with_sources.extend(search_history(query))

    if not context_chunks_with_sources:
        return "Not enough relevant context found."

    has_any_assistant = any(t["role"] == "assistant" for t in history)
    mode = "initial" if not has_any_assistant else "followup"

    prompt = build_prompt(history, context_chunks_with_sources, latest_user_message=query, mode=mode)

    return query_mistral(prompt)



In [13]:
q1="""From:
Anita Desai
Head of Digital Services
City of Riverton Council

Subject:
Request for Support on Citizen Services Modernization Initiative

Hi team,

We’re reaching out to explore working with your consultancy on a new initiative we're planning.

The City of Riverton is looking to modernize how we deliver public services online. Our current systems are fragmented and heavily manual — for example, applying for permits, reporting local issues, and accessing social support all happen on different platforms (some even still on paper).

Our aim is to create a unified digital experience for citizens. This would include:

    A central online portal where people can log in and access multiple services

    Backend automation to reduce manual workloads for our staff

    Better tracking and analytics to understand service usage and pain points

We’re particularly keen to learn what similar projects you’ve worked on with other public sector bodies, and how we can avoid common pitfalls. We know this will be a multi-phase project, and right now we’re just trying to get a clear picture of what the discovery and planning would look like.

Looking forward to hearing how you’d approach this.

Thanks,
Anita
    """


In [14]:
history=[]

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    response = rag(user_input, history)
    print(f"\nConsultant:\n{response}", flush=True)  # ensure timely print

    new_turns = [
        {"role": "user", "content": user_input},
        {"role": "assistant", "content": response},
    ]
    history.extend(new_turns)
    store_history_in_index(new_turns)



Consultant:
 - **Requested Client Requirements:**
  - Developing a proposal to integrate operational systems into a more digital estate.
  - Consistent support during the period leading to implementation.
  - Additional analysis and decision steering for Immediate Media leadership.
  - Setting up programme ways of working.
  - Introduction of digital delivery teams with a smooth mobilization process.
  - Following Clarasys Agile Methodology for delivery.
   (See [Proposals/Copy of FINAL ANSWERS TO BE REVIEWED FOR FORMATTING.docx] and [Proposals/Copy of Clarasys - Immediate Media Digital Transformation Mobilisation and Discovery proposal - Sept 2024.pptx])

- **Similar Past Projects:**
  - [Proposals/MASTER Business Architecture & Strategy Slides - Internal - Live .pptx]: This presentation discusses understanding the customer needs, channels for reaching segments, and phases of value creation. It also mentions examples such as assistance, self-service, communities, automated services, 